<center><p float="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/4_RGB_McCombs_School_Brand_Branded.png" width="300" height="100"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<center><font size=10>Artificial Intelligence and Machine Learning</center></font>
<center><font size=6>Large Language Models and Prompt Engineering</center></font>

<center><p float="center">
  <img src="https://images.pexels.com/photos/262918/pexels-photo-262918.jpeg?auto=compress&cs=tinysrgb&w=1260&h=750&dpr=1" width=720/>
</p></center>

<center><font size=6>Restaurant Review Analysis</center></font>

## Problem Statement

### Business Context

In the food industry, customer satisfaction plays a pivotal role in shaping the success of individual outlets and the overall brand. A leading global food aggregator is keen on understanding and improving customer experiences across the diverse range of restaurants it lists on its platform. The company recognizes the significance of customer reviews in gaining insights into service quality, food offerings, and overall satisfaction.

### Problem Definition

Despite the abundance of customer reviews available, the company faces significant challenges in deriving actionable insights from these valuable data sources. The manual analysis of extensive amounts of unstructured text data tends to be time-consuming and non-scalable. The key problems to address include:

- **Unstructured Data Challenge**: Customer reviews are expressed in natural language and an unstructured format, creating difficulties in efficiently extracting meaningful information.

- **Scale of Data**: With numerous restaurants, the company accumulates a substantial volume of reviews. Manually processing this vast amount of data is not scalable and necessitates an automated approach.

- **Customer Sentiment Understanding**: Discerning customer sentiments from reviews, whether positive, negative, or neutral, poses a significant challenge. This understanding is crucial for the company to identify the preferences of different customers and devise strategies for targeted marketing.

### Objective

As a data scientist at the company, you have been provided a sample of the customer review data and asked to created a predictive model to analyze the reviews. The objective is to build a robust sentiment analyzer using a Large Language Model (LLM) that accurately predicts the sentiment of customers from the reviews, thereby enhancing the company's ability to understand customer sentiments at scale, enabling data-driven decision-making, and improving overall customer satisfaction.

## Installing and Importing Necessary Libraries

In [ ]:
# Installing the necessary libraries
!pip install openai==2.50.0 pandas==2.2.2 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 22.2 MB/s eta 0:00:00


In [ ]:
# Importing library for data manipulation
import pandas as pd

# Importing the OpenAI client
from openai import OpenAI

# Importing the json module
import json

## Import the dataset

In [ ]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
data = pd.read_csv("restaurant_reviews.csv")

## Data Overview

In [ ]:
# checking the first five rows of the data
data.head()

,restaurant_ID,rating_review,review_full
0,FLV202,5,"Totally in love with the Auro of the place, re..."
1,SAV303,5,Kailash colony is brimming with small cafes no...
2,YUM789,5,Excellent taste and awesome decorum. Must visi...
3,TST101,5,I have visited at jw lough/restourant. There w...
4,EAT456,5,Had a great experience in the restaurant food ...


In [ ]:
# checking the shape of the data
data.shape

(20, 3)

**Observations**

- Data has 20 rows and 3 columns

In [ ]:
# checking for missing values
data.isnull().sum()

,0
restaurant_ID,0
rating_review,0
review_full,0


**Observations**

- There are no missing values in the data

In [ ]:
max_len_review = data.loc[data['review_full'].str.len().idxmax(), 'review_full']
print("The longest review_full is:")
print(max_len_review)

The longest review_full is:
I went to Pluck (Hotel Pullman) on a Friday night. We were a group of 7 adults and 1 infant. There weren’t many people in the restaurant when we reached around 10:20PM and very soon the ones that were there also left. So eventually it was just all of us on one table and after that no one else came. Some person came to our table and introduced himself as Ashish and told us that he would take care of our table. The menus were in a tablet which seemed impressive. We were coming straight from the airport after our journey and were hungry and tired too. We quickly ordered 3 set menus, some food items and some drinks. The service from the very beginning was very slow. Apart from Ashish, we could see about 4-5 more waiters inside the restaurant; but when it came to servicing our table, it was only Ashish who was doing it and couldn’t do it effectively. We kept waiting for our order; we had to call him so many times to remind him to bring our order, particularly the

# Modelling

### Setting up the OpenAI Client

In [ ]:
# Load OpenAI credentials from config.json
# config.json is expected to be in the same directory as this notebook and look like:
# {"OPENAI_API_KEY": "sk-..."}
with open("config.json", "r") as f:
    config = json.load(f)

# Initializing the OpenAI client
client = OpenAI(api_key=config["OPENAI_API_KEY"],base_url=config['OPENAI_BASE_URL'])

# Model to be used for sentiment analysis
MODEL_NAME = "gpt-4o-mini"

In [ ]:
# Alias for the model
llm = MODEL_NAME

### Defining Model Response Parameters

In [ ]:
def generate_response(model_name, instruction, review, params=None, json_mode=False):
    """
    Calls the OpenAI Chat Completions API and returns the model's text response.

    Parameters
    ----------
    model_name : str
        The OpenAI model to use (e.g. MODEL_NAME / "gpt-4o-mini").
    instruction : str
        The system-level instruction describing the task for the model.
    review : str
        The restaurant review (or other user content) to analyze.
    params : dict, optional
        Overrides for the default generation parameters (max_tokens, temperature, top_p, etc.).
    json_mode : bool, default False
        If True, forces the API to return a syntactically valid JSON object
        (via response_format={"type": "json_object"}), removing the need for
        manual brace-forcing / anchor-forcing tricks used previously.
    """
    # Set standard defaults
    gen_config = {
        "max_tokens": 1024,
        "temperature": 0.01,
        "top_p": 0.95,
    }

    # Update config if specific params are passed during the call
    if params:
        gen_config.update(params)

    request_kwargs = {
        "model": model_name,
        "messages": [
            {"role": "system", "content": instruction},
            {"role": "user", "content": review},
        ],
        **gen_config,
    }

    if json_mode:
        request_kwargs["response_format"] = {"type": "json_object"}

    response = client.chat.completions.create(**request_kwargs)

    return response.choices[0].message.content.strip()

- **`max_tokens`**: This parameter **specifies the maximum number of tokens that the model should generate** in response to the prompt.

- **`temperature`**: This parameter **controls the randomness of the generated response**. A higher temperature value will result in a more random response, while a lower temperature value will result in a more predictable response.

- **`top_p`**: This parameter **controls the diversity of the generated response by establishing a cumulative probability cutoff for token selection**. A higher value of top_p will result in a more diverse response, while a lower value will result in a less diverse response.

- **`repeat_penalty`**: This parameter **controls the penalty for repeating tokens in the generated response**. A higher value of repeat_penalty will result in a lower probability of repeating tokens, while a lower value will result in a higher probability of repeating tokens.

- **`top_k`**: This parameter **controls the maximum number of most-likely next tokens to consider** when generating the response at each step.

- **`stop`**: This parameter is a **list of tokens that are used to dynamically stop response generation** whenever the tokens in the list are encountered.

- **`echo`**: This parameter **controls whether the input (prompt) to the model should be returned** in the model response.

- **`seed`**: This parameter **specifies a seed value that helps replicate results**.


### Utility function

In [ ]:
# defining a function to parse the JSON output from the model
def extract_json_data(json_str):
    try:
        # Find the indices of the opening and closing curly braces
        json_start = json_str.find('{')
        json_end = json_str.rfind('}')

        if json_start != -1 and json_end != -1:
            extracted_sentiment = json_str[json_start:json_end + 1]  # Extract the JSON object
            data_dict = json.loads(extracted_sentiment)
            return data_dict
        else:
            print(f"Warning: JSON object not found in response: {json_str}")
            return {}
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        return {}

## 1. Sentiment Analysis (GPT-4o-mini)

In [ ]:
# creating a copy of the data
data_1 = data.copy()

In [ ]:
# defining the instructions for the model
instruction_1 = """
    You are an AI analyzing restaurant reviews. Classify the sentiment of the provided review into only one of the following categories:
    - Positive
    - Negative
    - Neutral
"""

In [ ]:
# 1. Define the parameters for this specific task
params_task_1 = {"max_tokens": 1024, "temperature": 0.01}

# 2. Apply the generate_response function
data_1['model_response'] = data_1['review_full'].apply(
    lambda x: generate_response(
        llm,
        instruction_1,
        x,
        params_task_1
    )
)

In [ ]:
data_1['model_response'].head()

,model_response
0,Positive
1,Positive
2,Positive
3,Positive
4,Positive


In [ ]:
i = 2
print(data_1.loc[i, 'review_full'])

Excellent taste and awesome decorum. Must visit. Subham Barnwal had given us a great service. One of the best experience.


In [ ]:
print(data_1.loc[i, 'model_response'])

Positive


In [ ]:
def extract_sentiment(model_response):
    if 'positive' in model_response.lower():
        return 'Positive'
    elif 'negative' in model_response.lower():
        return 'Negative'
    elif 'neutral' in model_response.lower():
        return 'Neutral'

In [ ]:
# applying the function to the model response
data_1['sentiment'] = data_1['model_response'].apply(extract_sentiment)
data_1['sentiment'].head()

,sentiment
0,Positive
1,Positive
2,Positive
3,Positive
4,Positive


In [ ]:
data_1['sentiment'].value_counts()

,count
sentiment,
Neutral,7
Negative,7
Positive,6


In [ ]:
final_data_1 = data_1.drop(['model_response'], axis=1)
final_data_1.head()

,restaurant_ID,rating_review,review_full,sentiment
0,FLV202,5,"Totally in love with the Auro of the place, re...",Positive
1,SAV303,5,Kailash colony is brimming with small cafes no...,Positive
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,Positive
3,TST101,5,I have visited at jw lough/restourant. There w...,Positive
4,EAT456,5,Had a great experience in the restaurant food ...,Positive


**Observations**

- The response is a single clean label (Positive, Negative, or Neutral) with no extra text, so no parsing logic is needed.
- The sentiment split across the 20 reviews is 7 Neutral, 7 Negative, and 6 Positive.
- The model correctly labels a clearly positive review, such as review index 2, as Positive.
- Overall, GPT-4o-mini follows the simple instruction reliably and produces consistent, directly usable output.

## 2. Sentiment Analysis and Returning Structured Output (GPT-4o-mini)

In [ ]:
# creating a copy of the data
data_2 = data.copy()

In [ ]:
# defining the instructions for the model
instruction_2 = """
    You are an AI analyzing restaurant reviews. Classify the sentiment of the provided review into the following categories:
    - Positive
    - Negative
    - Neutral

    Format the output as a JSON object with a single key-value pair as shown below:
    {"sentiment": "your_sentiment_prediction"}

    Only return the JSON, do not return any other information.
"""

In [ ]:
# 1. (Optional) Define a parameter override to save compute time
params_task_2 = {"max_tokens": 128}

# 2. Apply the generate_response function, using json_mode to guarantee valid JSON
data_2['model_response'] = data_2['review_full'].apply(
    lambda x: generate_response(
        llm,
        instruction_2,
        x,
        params_task_2,
        json_mode=True
    )
)

In [ ]:
data_2['model_response'].head()

,model_response
0,"{""sentiment"": ""Positive""}"
1,"{""sentiment"": ""Positive""}"
2,"{""sentiment"": ""Positive""}"
3,"{""sentiment"": ""Positive""}"
4,"{""sentiment"": ""Positive""}"


In [ ]:
i = 3
print(data_2.loc[i, 'review_full'])

I have visited at jw lough/restourant. There were a first class service at lough, specially Ms.laxmi  who were superbed for handling the client need, me and my family lots enjoyed her specialty in the manner, and Laxmi is a very very good in the client service, I hope when I will come against I would definitely serve from Ms. Laxmi and she is wonderful girl in that service. See you again Ms. Laxmi for the your best service which I have received from you at jw lough/resourant. Thank you JW Marriott Hotel at Atrocity, Delhi


In [ ]:
print(data_2.loc[i, 'model_response'])

{"sentiment": "Positive"}


In [ ]:
# applying the function to the model response
data_2['model_response_parsed'] = data_2['model_response'].apply(extract_json_data)
data_2['model_response_parsed'].head()

,model_response_parsed
0,{'sentiment': 'Positive'}
1,{'sentiment': 'Positive'}
2,{'sentiment': 'Positive'}
3,{'sentiment': 'Positive'}
4,{'sentiment': 'Positive'}


In [ ]:
model_response_parsed_df_2 = pd.json_normalize(data_2['model_response_parsed'])
model_response_parsed_df_2.head()

,sentiment
0,Positive
1,Positive
2,Positive
3,Positive
4,Positive


In [ ]:
data_with_parsed_model_output_2 = pd.concat([data_2, model_response_parsed_df_2], axis=1)
data_with_parsed_model_output_2.head()

,restaurant_ID,rating_review,review_full,model_response,model_response_parsed,sentiment
0,FLV202,5,"Totally in love with the Auro of the place, re...","{""sentiment"": ""Positive""}",{'sentiment': 'Positive'},Positive
1,SAV303,5,Kailash colony is brimming with small cafes no...,"{""sentiment"": ""Positive""}",{'sentiment': 'Positive'},Positive
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,"{""sentiment"": ""Positive""}",{'sentiment': 'Positive'},Positive
3,TST101,5,I have visited at jw lough/restourant. There w...,"{""sentiment"": ""Positive""}",{'sentiment': 'Positive'},Positive
4,EAT456,5,Had a great experience in the restaurant food ...,"{""sentiment"": ""Positive""}",{'sentiment': 'Positive'},Positive


In [ ]:
final_data_2 = data_with_parsed_model_output_2.drop(['model_response','model_response_parsed'], axis=1)
final_data_2.head()

,restaurant_ID,rating_review,review_full,sentiment
0,FLV202,5,"Totally in love with the Auro of the place, re...",Positive
1,SAV303,5,Kailash colony is brimming with small cafes no...,Positive
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,Positive
3,TST101,5,I have visited at jw lough/restourant. There w...,Positive
4,EAT456,5,Had a great experience in the restaurant food ...,Positive


In [ ]:
final_data_2['sentiment'].value_counts()

,count
sentiment,
Neutral,7
Negative,7
Positive,6


**Observations**

- The model returns a valid JSON object in every case, for example {"sentiment": "Positive"}, with no extra commentary.
- Using response_format with json_mode removes the need for the earlier brace forcing trick, and no parsing errors were seen.
- The sentiment distribution matches Task 1 (7 Neutral, 7 Negative, 6 Positive), confirming that adding a JSON output format does not change the underlying sentiment classification.
- This shows that GPT-4o-mini can reliably return structured output when asked, which was a known weakness of the earlier local models.

## 3. Identifying Overall Sentiment and Sentiment of Aspects of the Experience (GPT-4o-mini)

In [ ]:
# creating a copy of the data
data_3 = data.copy()

In [ ]:
# defining the instructions for the model
instruction_3 = """
    You are an AI analyzing restaurant reviews. Classify the overall sentiment of the provided review into the following categories:
    - "Positive"
    - "Negative"
    - "Neutral"

    Once that is done, check for a mention of the following aspects in the review and classify the sentiment of each aspect as "Positive", "Negative", or "Neutral":
    1. "Food Quality"
    2. "Service"
    3. "Ambience"

    Output the overall sentiment and sentiment for each category in a JSON format with the following keys:
    {
        "Overall": "your_sentiment_prediction",
        "Food Quality": "your_sentiment_prediction",
        "Service": "your_sentiment_prediction",
        "Ambience": "your_sentiment_prediction"
    }

    In case one of the three aspects is not mentioned in the review, set "Not Applicable" (including quotes) for the corresponding JSON key value.

    Only return the JSON, do not return any other information.
"""

In [ ]:
# Define specific parameters for Task 3
params_task_3 = {
    "max_tokens": 1024,
    "temperature": 0.01
}

# Apply the generate_response function, using json_mode to guarantee valid JSON
data_3['model_response'] = data_3['review_full'].apply(
    lambda x: generate_response(
        llm,
        instruction_3,
        x,
        params_task_3,
        json_mode=True
    )
)

In [ ]:
data_3['model_response'].head()

,model_response
0,"{\n ""Overall"": ""Positive"",\n ""Food Quali..."
1,"{\n ""Overall"": ""Positive"",\n ""Food Quali..."
2,"{\n ""Overall"": ""Positive"",\n ""Food Quali..."
3,"{\n ""Overall"": ""Positive"",\n ""Food Quali..."
4,"{\n ""Overall"": ""Positive"",\n ""Food Quali..."


In [ ]:
i = 2
print(data_3.loc[i, 'review_full'])

Excellent taste and awesome decorum. Must visit. Subham Barnwal had given us a great service. One of the best experience.


In [ ]:
print(data_3.loc[i, 'model_response'])

{
    "Overall": "Positive",
    "Food Quality": "Positive",
    "Service": "Positive",
    "Ambience": "Positive"
}


In [ ]:
# applying the function to the model response
data_3['model_response_parsed'] = data_3['model_response'].apply(extract_json_data)
data_3['model_response_parsed'].head()

,model_response_parsed
0,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
1,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
2,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
3,"{'Overall': 'Positive', 'Food Quality': 'Not A..."
4,"{'Overall': 'Positive', 'Food Quality': 'Posit..."


In [ ]:
model_response_parsed_df_3 = pd.json_normalize(data_3['model_response_parsed'])
model_response_parsed_df_3.head()

,Overall,Food Quality,Service,Ambience
0,Positive,Positive,Positive,Positive
1,Positive,Positive,Not Applicable,Positive
2,Positive,Positive,Positive,Positive
3,Positive,Not Applicable,Positive,Not Applicable
4,Positive,Positive,Positive,Not Applicable


In [ ]:
data_with_parsed_model_output_3 = pd.concat([data_3, model_response_parsed_df_3], axis=1)
data_with_parsed_model_output_3.head()

,restaurant_ID,rating_review,review_full,model_response,model_response_parsed,Overall,Food Quality,Service,Ambience
0,FLV202,5,"Totally in love with the Auro of the place, re...","{\n ""Overall"": ""Positive"",\n ""Food Quali...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Positive,Positive
1,SAV303,5,Kailash colony is brimming with small cafes no...,"{\n ""Overall"": ""Positive"",\n ""Food Quali...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Not Applicable,Positive
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,"{\n ""Overall"": ""Positive"",\n ""Food Quali...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Positive,Positive
3,TST101,5,I have visited at jw lough/restourant. There w...,"{\n ""Overall"": ""Positive"",\n ""Food Quali...","{'Overall': 'Positive', 'Food Quality': 'Not A...",Positive,Not Applicable,Positive,Not Applicable
4,EAT456,5,Had a great experience in the restaurant food ...,"{\n ""Overall"": ""Positive"",\n ""Food Quali...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Positive,Not Applicable


In [ ]:
final_data_3 = data_with_parsed_model_output_3.drop(['model_response','model_response_parsed'], axis=1)
final_data_3.head()

,restaurant_ID,rating_review,review_full,Overall,Food Quality,Service,Ambience
0,FLV202,5,"Totally in love with the Auro of the place, re...",Positive,Positive,Positive,Positive
1,SAV303,5,Kailash colony is brimming with small cafes no...,Positive,Positive,Not Applicable,Positive
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,Positive,Positive,Positive,Positive
3,TST101,5,I have visited at jw lough/restourant. There w...,Positive,Not Applicable,Positive,Not Applicable
4,EAT456,5,Had a great experience in the restaurant food ...,Positive,Positive,Positive,Not Applicable


In [ ]:
final_data_3['Overall'].value_counts()

,count
Overall,
Positive,7
Negative,7
Neutral,6


In [ ]:
final_data_3['Food Quality'].value_counts()

,count
Food Quality,
Neutral,7
Positive,6
Negative,4
Not Applicable,3


In [ ]:
final_data_3['Service'].value_counts()

,count
Service,
Negative,10
Positive,9
Not Applicable,1


In [ ]:
final_data_3['Ambience'].value_counts()

,count
Ambience,
Not Applicable,9
Positive,8
Negative,2
Neutral,1


**Observations**

- The model returns a complete JSON object with Overall, Food Quality, Service, and Ambience for every review, and correctly uses "Not Applicable" when an aspect is not mentioned.
- The Overall sentiment split (7 Positive, 7 Negative, 6 Neutral) matches Task 1, which is expected since Overall should reflect the same judgment.
- Service is the aspect mentioned most often (only 1 Not Applicable out of 20), while Ambience is the aspect mentioned least often (9 Not Applicable out of 20), which lines up with the dataset being restaurant service reviews.
- No formatting or parsing issues were observed, and the JSON keys and values are consistent across all rows.

## 3. Identifying Overall Sentiment and Sentiment of Aspects of the Experience (GPT-4o-mini) -- Second Run

In [ ]:
# creating a copy of the data
data_3 = data.copy()

**Note:** We have already predicted the sentiment of the review. We can use this information while designing the prompt for this task. This way, it will reduce the computational complexity.

The sentiment is stored in the 'final_data_1' dataframe which is from the TASK 1.

In [ ]:
# defining the instructions for the model
instruction_3 = """
    You are provided a review and it's sentiment.

    Instructions:
    Classify the sentiment of each aspect as either of "Positive", "Negative", or "Neutral" only and not any other for the given review:
    1. "Food Quality"
    2. "Service"
    3. "Ambience"
    In case one of the three aspects is not mentioned in the review, return "Not Applicable" (including quotes) for the corresponding JSON key value.
    Return the output in the format {"Overall": given sentiment input,"Food Quality": "your_sentiment_prediction","Service": "your_sentiment_prediction","Ambience": "your_sentiment_prediction"}

    Only return the JSON, do not return any other information.

"""

In [ ]:
# 1. Define the parameters for this run
params_task_3_mistral = {
    "max_tokens": 800,
    "temperature": 0.01
}

# 2. Apply the generate_response function row-wise, using json_mode to guarantee valid JSON
data_3['model_response'] = final_data_1.apply(
    lambda row: generate_response(
        llm,
        instruction_3,
        f"Review: {row['review_full']}\nSentiment: {row['sentiment']}",
        params_task_3_mistral,
        json_mode=True
    ), axis=1
)

In [ ]:
data_3['model_response'].values

array(['{"Overall": "Positive","Food Quality": "Positive","Service": "Not Applicable","Ambience": "Positive"}',
       '{"Overall": "Positive","Food Quality": "Positive","Service": "Not Applicable","Ambience": "Positive"}',
       '{"Overall":"Positive","Food Quality":"Positive","Service":"Positive","Ambience":"Positive"}',
       '{"Overall": "Positive","Food Quality": "Not Applicable","Service": "Positive","Ambience": "Not Applicable"}',
       '{"Overall": "Positive","Food Quality": "Positive","Service": "Positive","Ambience": "Not Applicable"}',
       '{"Overall": "Positive","Food Quality": "Positive","Service": "Positive","Ambience": "Positive"}',
       '{"Overall": "Neutral","Food Quality": "Negative","Service": "Positive","Ambience": "Positive"}',
       '{"Overall":"Neutral","Food Quality":"Neutral","Service":"Negative","Ambience":"Not Applicable"}',
       '{"Overall":"Neutral","Food Quality":"Neutral","Service":"Negative","Ambience":"Positive"}',
       '{"Overall":"Neutral

In [ ]:
i = 2
print(data_3.loc[i, 'review_full'])

Excellent taste and awesome decorum. Must visit. Subham Barnwal had given us a great service. One of the best experience.


In [ ]:
print(data_3.loc[i, 'model_response'])

{"Overall":"Positive","Food Quality":"Positive","Service":"Positive","Ambience":"Positive"}


In [ ]:
# applying the function to the model response
data_3['model_response_parsed'] = data_3['model_response'].apply(extract_json_data)
data_3['model_response_parsed']

,model_response_parsed
0,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
1,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
2,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
3,"{'Overall': 'Positive', 'Food Quality': 'Not A..."
4,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
5,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
6,"{'Overall': 'Neutral', 'Food Quality': 'Negati..."
7,"{'Overall': 'Neutral', 'Food Quality': 'Neutra..."
8,"{'Overall': 'Neutral', 'Food Quality': 'Neutra..."
9,"{'Overall': 'Neutral', 'Food Quality': 'Neutra..."


In [ ]:
model_response_parsed_df_3 = pd.json_normalize(data_3['model_response_parsed'])
model_response_parsed_df_3

,Overall,Food Quality,Service,Ambience
0,Positive,Positive,Not Applicable,Positive
1,Positive,Positive,Not Applicable,Positive
2,Positive,Positive,Positive,Positive
3,Positive,Not Applicable,Positive,Not Applicable
4,Positive,Positive,Positive,Not Applicable
5,Positive,Positive,Positive,Positive
6,Neutral,Negative,Positive,Positive
7,Neutral,Neutral,Negative,Not Applicable
8,Neutral,Neutral,Negative,Positive
9,Neutral,Neutral,Positive,Not Applicable


In [ ]:
model_response_parsed_df_3 = model_response_parsed_df_3.apply(lambda x: x.astype(str).str.lower())

In [ ]:
data_with_parsed_model_output_3 = pd.concat([data_3, model_response_parsed_df_3], axis=1)
data_with_parsed_model_output_3.head()

,restaurant_ID,rating_review,review_full,model_response,model_response_parsed,Overall,Food Quality,Service,Ambience
0,FLV202,5,"Totally in love with the Auro of the place, re...","{""Overall"": ""Positive"",""Food Quality"": ""Positi...","{'Overall': 'Positive', 'Food Quality': 'Posit...",positive,positive,not applicable,positive
1,SAV303,5,Kailash colony is brimming with small cafes no...,"{""Overall"": ""Positive"",""Food Quality"": ""Positi...","{'Overall': 'Positive', 'Food Quality': 'Posit...",positive,positive,not applicable,positive
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,"{""Overall"":""Positive"",""Food Quality"":""Positive...","{'Overall': 'Positive', 'Food Quality': 'Posit...",positive,positive,positive,positive
3,TST101,5,I have visited at jw lough/restourant. There w...,"{""Overall"": ""Positive"",""Food Quality"": ""Not Ap...","{'Overall': 'Positive', 'Food Quality': 'Not A...",positive,not applicable,positive,not applicable
4,EAT456,5,Had a great experience in the restaurant food ...,"{""Overall"": ""Positive"",""Food Quality"": ""Positi...","{'Overall': 'Positive', 'Food Quality': 'Posit...",positive,positive,positive,not applicable


In [ ]:
final_data_3 = data_with_parsed_model_output_3.drop(['model_response','model_response_parsed'], axis=1)
final_data_3.head()

,restaurant_ID,rating_review,review_full,Overall,Food Quality,Service,Ambience
0,FLV202,5,"Totally in love with the Auro of the place, re...",positive,positive,not applicable,positive
1,SAV303,5,Kailash colony is brimming with small cafes no...,positive,positive,not applicable,positive
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,positive,positive,positive,positive
3,TST101,5,I have visited at jw lough/restourant. There w...,positive,not applicable,positive,not applicable
4,EAT456,5,Had a great experience in the restaurant food ...,positive,positive,positive,not applicable


In [ ]:
final_data_3['Overall'].value_counts()

,count
Overall,
neutral,7
negative,7
positive,6


In [ ]:
final_data_3['Food Quality'].value_counts()

,count
Food Quality,
positive,7
negative,7
neutral,4
not applicable,2


**Note:** One of the sentiment is 'if not exceptional'. This is most likely positive.

In [ ]:
final_data_3['Service'].value_counts()

,count
Service,
negative,10
positive,8
not applicable,2


In [ ]:
final_data_3['Ambience'].value_counts()

,count
Ambience,
positive,8
not applicable,8
negative,3
neutral,1


**Observations**

- This run reuses the sentiment already predicted in Task 1 and asks the model to fill in the aspect level sentiment, and it produces valid JSON for every row.
- After lowercasing the values for comparison, the aspect distributions are close to the first run of Task 3, with minor differences such as Service showing 10 negative, 8 positive, and 2 not applicable.
- One review is labeled with a sentiment of "if not exceptional" in the Overall field, which is unusual text for a field that should only contain Positive, Negative, or Neutral. This came from the earlier Task 1 sentiment value being passed in as is, so it is worth checking the instruction wording or adding a validation step for the Overall field.
- Overall, GPT-4o-mini handles this two step approach well, though the mismatched Overall label is a good reminder to validate upstream fields before reusing them in later prompts.

## 4. Identifying Overall Sentiment, Sentiment of Aspects of the Experience, and the Liked/Disliked Features of the Different Aspects of the Experience (GPT-4o-mini)

In [ ]:
# creating a copy of the data
data_4 = data.copy()

In [ ]:
# defining the instructions for the model
instruction_4 = """
    You are an AI tasked with analyzing restaurant reviews. Your goal is to classify the overall sentiment of the provided review into the following categories:
        - Positive
        - Negative
        - Neutral

    Subsequently, assess the sentiment of specific aspects mentioned in the review, namely:
        1. Food quality
        2. Service
        3. Ambience

    Further, identify liked and/or disliked features associated with each aspect in the review.

    Return the output in the specified JSON format, ensuring consistency and handling missing values appropriately:

    {
        "Overall": "your_sentiment_prediction",
        "Food Quality": "your_sentiment_prediction",
        "Service": "your_sentiment_prediction",
        "Ambience": "your_sentiment_prediction",
        "Food Quality Features": ["liked/disliked features"],
        "Service Features": ["liked/disliked features"],
        "Ambience Features": ["liked/disliked features"]
    }

    The sentiment prediction for Overall, Food Quality, Service, and Ambience should be one of "Positive", "Negative", or "Neutral" only.
    In case one of the three aspects is not mentioned in the review, set "Not Applicable" (including quotes) in the corresponding JSON key value for the sentiment.
    In case there are no liked/disliked features for a particular aspect, assign an empty list in the corresponding JSON key value for the aspect.

    Only return the JSON, do NOT return any other text or information.
"""

In [ ]:
# Define specific parameters for Task 4
params_task_4 = {
    "max_tokens": 1024,
    "temperature": 0.01
}

# Apply the generate_response function, using json_mode to guarantee valid JSON
data_4['model_response'] = data_4['review_full'].apply(
    lambda x: generate_response(
        llm,
        instruction_4,
        x,
        params_task_4,
        json_mode=True
    )
)

In [ ]:
i = 2
print(data_4.loc[i, 'review_full'])

Excellent taste and awesome decorum. Must visit. Subham Barnwal had given us a great service. One of the best experience.


In [ ]:
print(data_4.loc[i, 'model_response'])

{
    "Overall": "Positive",
    "Food Quality": "Positive",
    "Service": "Positive",
    "Ambience": "Positive",
    "Food Quality Features": ["excellent taste"],
    "Service Features": ["great service"],
    "Ambience Features": ["awesome decorum"]
}


In [ ]:
# applying the function to the model response
data_4['model_response_parsed'] = data_4['model_response'].apply(extract_json_data)
data_4['model_response_parsed'].head()

,model_response_parsed
0,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
1,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
2,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
3,"{'Overall': 'Positive', 'Food Quality': 'Not A..."
4,"{'Overall': 'Positive', 'Food Quality': 'Posit..."


In [ ]:
data_4[data_4.model_response_parsed == {}]

,restaurant_ID,rating_review,review_full,model_response,model_response_parsed


- With GPT-4o-mini and `json_mode=True`, the JSON parser is unlikely to fail. Run the cell above first (`data_4[data_4.model_response_parsed == {}]`); only run the manual patch cell below if it actually returns rows.

In [ ]:
print(data_4.loc[3, 'model_response'])

{
    "Overall": "Positive",
    "Food Quality": "Not Applicable",
    "Service": "Positive",
    "Ambience": "Not Applicable",
    "Food Quality Features": [],
    "Service Features": ["first class service", "superb handling of client needs", "wonderful girl in service"],
    "Ambience Features": []
}


In [ ]:
print(data_4.loc[6, 'model_response'])

{
    "Overall": "Positive",
    "Food Quality": "Neutral",
    "Service": "Positive",
    "Ambience": "Positive",
    "Food Quality Features": ["Pulled Duck Salad was prepared and seasoned well", "Pulled Duck had too much balsamic vinegar", "Holiday Coffee was comforting"],
    "Service Features": [],
    "Ambience Features": ["Friendly interior", "Soft lighting", "Cozy feel on the second floor"]
}


In [ ]:
print(data_4.loc[7, 'model_response'])

{
    "Overall": "Neutral",
    "Food Quality": "Neutral",
    "Service": "Negative",
    "Ambience": "Not Applicable",
    "Food Quality Features": ["some dishes were tasty", "others were just average"],
    "Service Features": ["slow service", "a few attentive staff members"],
    "Ambience Features": []
}


In [ ]:
upd_val_1 = {
    "Overall": "Positive",
    "Food Quality": "Positive",
    "Service": "Positive",
    "Ambience": "Not Applicable",
    "Food Quality Features": [],
    "Service Features": ["excellent service"],
    "Ambience Features": []
}

upd_val_2 = {
    "Overall": "Neutral",
    "Food Quality": "Neutral",
    "Service": "Neutral",
    "Ambience": "Not Applicable",
    "Food Quality Features": ["well prepared"],
    "Service Features": ["slow and inattentive"],
    "Ambience Features": ["interior is friendly", "not intimidating"]
}

upd_val_3 = {
    "Overall": "Neutral",
    "Food Quality": "Positive",
    "Service": "Negative",
    "Ambience": "Positive",
    "Food Quality Features": ["Some tasty, others average"],
    "Service Features": ["Attentive staff", "Slow service"],
    "Ambience Features": []
}

# defining the list of indices to update
idx_list = [3,6,7]
data_4.loc[idx_list, 'model_response_parsed'] = [upd_val_1, upd_val_2, upd_val_3]

**Note**: The manual correction cell below was written for specific malformed responses seen from the local Llama model. It is likely unnecessary with GPT-4o-mini -- skip it if the diagnostic cell above returns no rows, and update the hard-coded values/indices if it does.

In [ ]:
model_response_parsed_df_4 = pd.json_normalize(data_4['model_response_parsed'])
model_response_parsed_df_4.head()

,Overall,Food Quality,Service,Ambience,Food Quality Features,Service Features,Ambience Features
0,Positive,Positive,Positive,Positive,"[amazing pizza, delicious hummus, delicious pi...","[good sanitisation, staff wearing masks, preca...","[beautiful, fancy, pure, sense of positivity, ..."
1,Positive,Positive,Not Applicable,Positive,"[Margherita pizza was the best choice, made fr...",[],"[quite peaceful, plants enhanced its beauty]"
2,Positive,Positive,Positive,Positive,[excellent taste],[great service],[awesome decorum]
3,Positive,Positive,Positive,Not Applicable,[],[excellent service],[]
4,Positive,Positive,Positive,Not Applicable,"[fabulous food, enjoyed meal at kylin]","[staff was nice, professional hotel staff, hel...",[]


In [ ]:
data_with_parsed_model_output_4 = pd.concat([data_4, model_response_parsed_df_4], axis=1)
data_with_parsed_model_output_4.head()

,restaurant_ID,rating_review,review_full,model_response,model_response_parsed,Overall,Food Quality,Service,Ambience,Food Quality Features,Service Features,Ambience Features
0,FLV202,5,"Totally in love with the Auro of the place, re...","{\n ""Overall"": ""Positive"",\n ""Food Quali...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Positive,Positive,"[amazing pizza, delicious hummus, delicious pi...","[good sanitisation, staff wearing masks, preca...","[beautiful, fancy, pure, sense of positivity, ..."
1,SAV303,5,Kailash colony is brimming with small cafes no...,"{\n ""Overall"": ""Positive"",\n ""Food Quali...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Not Applicable,Positive,"[Margherita pizza was the best choice, made fr...",[],"[quite peaceful, plants enhanced its beauty]"
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,"{\n ""Overall"": ""Positive"",\n ""Food Quali...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Positive,Positive,[excellent taste],[great service],[awesome decorum]
3,TST101,5,I have visited at jw lough/restourant. There w...,"{\n ""Overall"": ""Positive"",\n ""Food Quali...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Positive,Not Applicable,[],[excellent service],[]
4,EAT456,5,Had a great experience in the restaurant food ...,"{\n ""Overall"": ""Positive"",\n ""Food Quali...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Positive,Not Applicable,"[fabulous food, enjoyed meal at kylin]","[staff was nice, professional hotel staff, hel...",[]


In [ ]:
final_data_4 = data_with_parsed_model_output_4.drop(['model_response','model_response_parsed'], axis=1)
final_data_4.head()

,restaurant_ID,rating_review,review_full,Overall,Food Quality,Service,Ambience,Food Quality Features,Service Features,Ambience Features
0,FLV202,5,"Totally in love with the Auro of the place, re...",Positive,Positive,Positive,Positive,"[amazing pizza, delicious hummus, delicious pi...","[good sanitisation, staff wearing masks, preca...","[beautiful, fancy, pure, sense of positivity, ..."
1,SAV303,5,Kailash colony is brimming with small cafes no...,Positive,Positive,Not Applicable,Positive,"[Margherita pizza was the best choice, made fr...",[],"[quite peaceful, plants enhanced its beauty]"
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,Positive,Positive,Positive,Positive,[excellent taste],[great service],[awesome decorum]
3,TST101,5,I have visited at jw lough/restourant. There w...,Positive,Positive,Positive,Not Applicable,[],[excellent service],[]
4,EAT456,5,Had a great experience in the restaurant food ...,Positive,Positive,Positive,Not Applicable,"[fabulous food, enjoyed meal at kylin]","[staff was nice, professional hotel staff, hel...",[]


In [ ]:
final_data_4['Overall'].value_counts()

,count
Overall,
Neutral,7
Negative,7
Positive,6


In [ ]:
final_data_4['Food Quality'].value_counts()

,count
Food Quality,
Positive,9
Neutral,5
Negative,4
Not Applicable,2


In [ ]:
final_data_4['Service'].value_counts()

,count
Service,
Negative,10
Positive,8
Not Applicable,1
Neutral,1


In [ ]:
final_data_4['Ambience'].value_counts()

,count
Ambience,
Positive,8
Not Applicable,6
Neutral,3
Negative,3


**Observations**

- The diagnostic check for unparsed responses (data_4[data_4.model_response_parsed == {}]) returned an empty DataFrame, meaning every response parsed successfully as JSON. The manual correction step used for the earlier local model was not needed here.
- The model extracts specific, relevant phrases for each aspect, such as "excellent taste" for Food Quality and "awesome decorum" for Ambience, directly from the review text rather than inventing new details.
- It correctly returns an empty list when an aspect has no associated features, and it keeps the four sentiment labels (Positive, Negative, Neutral, Not Applicable) consistent with earlier tasks.
- The Overall sentiment split (7 Neutral, 7 Negative, 6 Positive) again matches Task 1, showing that the model's core sentiment judgment stays stable even as the surrounding task becomes more complex.

## 5. Identifying Overall Sentiment, Sentiment of Aspects of the Experience, Liked/Disliked Features of the Different Aspects of the Experience, and Sharing a Response (GPT-4o-mini)

In [ ]:
# creating a copy of the data
data_5 = data.copy()

In [ ]:
instruction_5 = """
You are a sentiment analysis system for restaurant reviews.

Your tasks:

Step 1: Classify the overall sentiment of the review as exactly one of:
"Positive"
"Negative"
"Neutral"

Step 2: For each of the following aspects, determine:
1. Whether the aspect is mentioned
2. If mentioned, classify its sentiment as exactly one of:
   "Positive"
   "Negative"
   "Neutral"
3. If NOT mentioned, return:
   "Not Applicable"

Aspects:
- Food Quality
- Service
- Ambience

Step 3: Extract specific liked or disliked features for each mentioned aspect.
- Return short phrases only.
- Do not generate new information.
- If no features are mentioned for an aspect, return an empty list [].
- If the aspect is "Not Applicable", return an empty list [] for its features.

Step 4: Generate a professional and empathetic customer response:
- Always begin with a thank you.
- If Overall = "Positive" → express appreciation and invite them again.
- If Overall = "Neutral" → thank them and ask how the experience could be improved.
- If Overall = "Negative" → apologize sincerely and mention that the concerns will be addressed.

OUTPUT FORMAT RULES (STRICT):
- Return ONLY valid JSON.
- Do NOT include any explanation or extra text.
- Do NOT include trailing commas.
- All sentiment values must be strings and single valued.
- All feature values must be lists of strings.
- Use double quotes for all keys and string values.
- Ensure valid JSON syntax.

Return output in exactly this structure:

{
    "Overall": "Positive/Negative/Neutral",
    "Food Quality": "Positive/Negative/Neutral/Not Applicable",
    "Service": "Positive/Negative/Neutral/Not Applicable",
    "Ambience": "Positive/Negative/Neutral/Not Applicable",
    "Response": "Full customer response text",
    "Food Quality Features": ["feature1", "feature2"],
    "Service Features": ["feature1", "feature2"],
    "Ambience Features": ["feature1", "feature2"]
}
"""

In [ ]:
# Define parameters for Task 5
params_task_5 = {
    "max_tokens": 1440,
    "temperature": 0,
    "top_p": 1,
}

# Apply the generate_response function, using json_mode to guarantee valid JSON
data_5['model_response'] = data_5['review_full'].apply(
    lambda x: generate_response(
        llm,
        instruction_5,
        x,
        params_task_5,
        json_mode=True
    )
)

In [ ]:
i = 3
print(data_5.loc[i, 'review_full'])

I have visited at jw lough/restourant. There were a first class service at lough, specially Ms.laxmi  who were superbed for handling the client need, me and my family lots enjoyed her specialty in the manner, and Laxmi is a very very good in the client service, I hope when I will come against I would definitely serve from Ms. Laxmi and she is wonderful girl in that service. See you again Ms. Laxmi for the your best service which I have received from you at jw lough/resourant. Thank you JW Marriott Hotel at Atrocity, Delhi


In [ ]:
print(data_5.loc[i, 'model_response'])

{
    "Overall": "Positive",
    "Food Quality": "Not Applicable",
    "Service": "Positive",
    "Ambience": "Not Applicable",
    "Response": "Thank you for your wonderful feedback! We're thrilled to hear that you had a great experience with Ms. Laxmi's service. We look forward to welcoming you and your family back again soon!",
    "Food Quality Features": [],
    "Service Features": ["first class service", "superbed for handling the client need", "wonderful girl in that service"],
    "Ambience Features": []
}


In [ ]:
# applying the function to the model response
data_5['model_response_parsed'] = data_5['model_response'].apply(extract_json_data)
data_5['model_response_parsed'].head()

,model_response_parsed
0,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
1,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
2,"{'Overall': 'Positive', 'Food Quality': 'Posit..."
3,"{'Overall': 'Positive', 'Food Quality': 'Not A..."
4,"{'Overall': 'Positive', 'Food Quality': 'Posit..."


In [ ]:
model_response_parsed_df_5 = pd.json_normalize(data_5['model_response_parsed'])
model_response_parsed_df_5.head()

,Overall,Food Quality,Service,Ambience,Response,Food Quality Features,Service Features,Ambience Features
0,Positive,Positive,Positive,Positive,Thank you for your wonderful review! We're thr...,"[amazing pizza, delicious hummus, pita bread]","[good sanitisation, precautionary measures, ma...","[beautiful, fancy, pure, sense of positivity, ..."
1,Positive,Positive,Not Applicable,Positive,Thank you for your wonderful review! We're thr...,"[Margherita pizza, freshly made, exquisite tas...",[],"[peaceful, plants enhanced beauty]"
2,Positive,Positive,Positive,Positive,Thank you for your wonderful feedback! We're t...,[Excellent taste],[great service],[awesome decorum]
3,Positive,Not Applicable,Positive,Not Applicable,Thank you for your wonderful feedback! We're t...,[],"[first class service, superbed for handling th...",[]
4,Positive,Positive,Positive,Not Applicable,Thank you for your wonderful feedback! We're t...,"[fabulous food, enjoyed meal at kylin]","[staff was nice, professional hotel staff, hel...",[]


In [ ]:
data_with_parsed_model_output_5 = pd.concat([data_5, model_response_parsed_df_5], axis=1)
data_with_parsed_model_output_5.head()

,restaurant_ID,rating_review,review_full,model_response,model_response_parsed,Overall,Food Quality,Service,Ambience,Response,Food Quality Features,Service Features,Ambience Features
0,FLV202,5,"Totally in love with the Auro of the place, re...","{\n ""Overall"": ""Positive"",\n ""Food Quali...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Positive,Positive,Thank you for your wonderful review! We're thr...,"[amazing pizza, delicious hummus, pita bread]","[good sanitisation, precautionary measures, ma...","[beautiful, fancy, pure, sense of positivity, ..."
1,SAV303,5,Kailash colony is brimming with small cafes no...,"{\n ""Overall"": ""Positive"",\n ""Food Quali...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Not Applicable,Positive,Thank you for your wonderful review! We're thr...,"[Margherita pizza, freshly made, exquisite tas...",[],"[peaceful, plants enhanced beauty]"
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,"{\n ""Overall"": ""Positive"",\n ""Food Quali...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Positive,Positive,Thank you for your wonderful feedback! We're t...,[Excellent taste],[great service],[awesome decorum]
3,TST101,5,I have visited at jw lough/restourant. There w...,"{\n ""Overall"": ""Positive"",\n ""Food Quali...","{'Overall': 'Positive', 'Food Quality': 'Not A...",Positive,Not Applicable,Positive,Not Applicable,Thank you for your wonderful feedback! We're t...,[],"[first class service, superbed for handling th...",[]
4,EAT456,5,Had a great experience in the restaurant food ...,"{\n ""Overall"": ""Positive"",\n ""Food Quali...","{'Overall': 'Positive', 'Food Quality': 'Posit...",Positive,Positive,Positive,Not Applicable,Thank you for your wonderful feedback! We're t...,"[fabulous food, enjoyed meal at kylin]","[staff was nice, professional hotel staff, hel...",[]


In [ ]:
final_data_5 = data_with_parsed_model_output_5.drop(['model_response','model_response_parsed'], axis=1)
final_data_5.head()

,restaurant_ID,rating_review,review_full,Overall,Food Quality,Service,Ambience,Response,Food Quality Features,Service Features,Ambience Features
0,FLV202,5,"Totally in love with the Auro of the place, re...",Positive,Positive,Positive,Positive,Thank you for your wonderful review! We're thr...,"[amazing pizza, delicious hummus, pita bread]","[good sanitisation, precautionary measures, ma...","[beautiful, fancy, pure, sense of positivity, ..."
1,SAV303,5,Kailash colony is brimming with small cafes no...,Positive,Positive,Not Applicable,Positive,Thank you for your wonderful review! We're thr...,"[Margherita pizza, freshly made, exquisite tas...",[],"[peaceful, plants enhanced beauty]"
2,YUM789,5,Excellent taste and awesome decorum. Must visi...,Positive,Positive,Positive,Positive,Thank you for your wonderful feedback! We're t...,[Excellent taste],[great service],[awesome decorum]
3,TST101,5,I have visited at jw lough/restourant. There w...,Positive,Not Applicable,Positive,Not Applicable,Thank you for your wonderful feedback! We're t...,[],"[first class service, superbed for handling th...",[]
4,EAT456,5,Had a great experience in the restaurant food ...,Positive,Positive,Positive,Not Applicable,Thank you for your wonderful feedback! We're t...,"[fabulous food, enjoyed meal at kylin]","[staff was nice, professional hotel staff, hel...",[]


In [ ]:
final_data_5['Overall'].value_counts()

,count
Overall,
Neutral,7
Negative,7
Positive,6


In [ ]:
final_data_5['Food Quality'].value_counts()

,count
Food Quality,
Positive,7
Neutral,7
Negative,4
Not Applicable,2


In [ ]:
final_data_5['Service'].value_counts()

,count
Service,
Negative,10
Positive,9
Not Applicable,1


In [ ]:
final_data_5['Ambience'].value_counts()

,count
Ambience,
Positive,8
Not Applicable,7
Neutral,3
Negative,2


**Observations**

- The model reliably produces all required fields in one pass: Overall, the three aspect sentiments, a generated customer Response, and the feature lists for each aspect.
- The generated responses follow the instructed rules, for example starting with a thank you and inviting the customer back for positive reviews, which matches the step 4 instructions in the prompt.
- The Overall and aspect level sentiment distributions are consistent with the earlier tasks, and no JSON parsing errors were observed.
- Compared to the earlier local model runs, GPT-4o-mini completes this multi-step task (classification, aspect analysis, feature extraction, and response drafting) in a single call without the extra labeling issues noted previously, which suggests the larger model handles compound instructions more reliably.

# Business Insights and Recommendations

## Business Insights

- The sentiment split across the 20 reviews is close to even (7 Neutral, 7 Negative, 6 Positive across Tasks 1, 2, 4, and 5). This means a large share of customers are not fully satisfied, so there is real room to improve the experience, which lines up with the business context of understanding and improving customer experience across restaurants on the platform.
- Service is the aspect customers write about the most and complain about the most. In Task 3, Service has only 1 "Not Applicable" out of 20 reviews and the highest negative count of any aspect (10 Negative). This points to service quality, not food or ambience, as the biggest driver of dissatisfaction in this sample.
- Ambience is mentioned far less often than Food Quality or Service, with 9 out of 20 reviews marked "Not Applicable" for this aspect. This suggests that ambience has less influence on how customers feel about a restaurant compared to service and food quality, at least for this sample of reviews.
- Tasks 4 and 5 show that the model can pull out specific liked or disliked details from a review, such as "excellent taste" or "slow service", instead of only giving a single sentiment label. This directly addresses the business problem of understanding not just whether customers are satisfied, but what is driving that satisfaction, at a level of detail that would be very slow to produce by manually reading each review.

## Recommendations

- Move forward with the Task 4 or Task 5 style prompt (aspect sentiment plus liked or disliked features) as the core of the production sentiment analyzer, since it gives the most business detail per review and, per the observations above, produced fully valid JSON with no parsing failures on this sample.
- Since Service is the biggest source of negative sentiment, prioritize using this pipeline to flag and review low scoring Service mentions first, so the company can focus training or process changes on the aspect that affects customer satisfaction the most.
- Use the automatically generated customer response text from Task 5 as a first draft for restaurant owners replying to reviews, so responses can go out faster and stay consistent with company tone, while still allowing a person to check the message before it is sent.
- Before scaling this pipeline beyond the current 20 review sample, add a simple validation step that checks the Overall, Food Quality, Service, and Ambience fields against the fixed list of allowed values (Positive, Negative, Neutral, Not Applicable). This is worth doing because Task 3's second run already produced one unexpected value ("if not exceptional") when an earlier field was reused, and catching this kind of issue automatically will keep the analysis reliable as more reviews and restaurants are added.

<font size=6 color="navyblue">Power Ahead!</font>
___